In [ ]:
import cv2
import os
import glob
import numpy as np
from wisardpkg import ClusWisard
import sys
sys.path.append('/mnt/c/Users/Isabella/tcc')
from wisard_object_tracker.src.utils import tracker_utils
import matplotlib.pyplot as plt

DATASET_ROOT_FOLDER = '/mnt/c/Users/Isabella/TCC/wisard_object_tracker/data/tiger2'
IMAGE_FOLDER = os.path.join(DATASET_ROOT_FOLDER, 'imgs')
GT_TXT_PATH = os.path.join(DATASET_ROOT_FOLDER, 'tiger2_gt.txt')

In [ ]:
# Carrega imagens
image_paths = sorted(glob.glob(os.path.join(IMAGE_FOLDER, '*.png')))
print(f"Encontradas {len(image_paths)} imagens")

# Carrega ground truths
all_ground_truths = tracker_utils.load_ground_truth_from_gt_txt(GT_TXT_PATH)

In [ ]:
import cv2
import os
import glob
import numpy as np
import matplotlib.pyplot as plt

DATASET_ROOT_FOLDER = '/mnt/c/Users/Isabella/TCC/wisard_object_tracker/data/tiger2'
IMAGE_FOLDER = os.path.join(DATASET_ROOT_FOLDER, 'imgs')

# --------------------------------------------------
# Carrega imagens
# --------------------------------------------------
image_paths = sorted(glob.glob(os.path.join(IMAGE_FOLDER, '*.png')))
print(f"Encontradas {len(image_paths)} imagens")

# --------------------------------------------------
# Estima background fixo (mediana)
# --------------------------------------------------
bg_frames = []

for p in image_paths[:1]:
    img = cv2.imread(p)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    bg_frames.append(gray)

background = np.median(bg_frames, axis=0).astype(np.uint8)
print("✅ Background fixo estimado")

# --------------------------------------------------
# Parâmetros
# --------------------------------------------------
kernel = np.ones((5, 5), np.uint8)

# --------------------------------------------------
# LOOP EM TODOS OS FRAMES
# --------------------------------------------------
for idx, img_path in enumerate(image_paths):

    frame = cv2.imread(img_path)
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Diferença absoluta
    diff = cv2.absdiff(frame_gray, background)

    # Threshold automático (Otsu)
    _, fg_mask = cv2.threshold(
        diff,
        0,
        255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    # Limpeza morfológica
    fg_mask = cv2.morphologyEx(fg_mask, cv2.MORPH_OPEN, kernel)

    # Aplica máscara
    fg_frame = cv2.bitwise_and(frame_rgb, frame_rgb, mask=fg_mask)

    # --------------------------------------------------
    # Visualização
    # --------------------------------------------------
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))

    ax[0].imshow(frame_rgb)
    ax[0].set_title(f"Frame {idx} - Original")
    ax[0].axis("off")

    ax[1].imshow(fg_mask, cmap="gray")
    ax[1].set_title("Máscara foreground (Otsu)")
    ax[1].axis("off")

    ax[2].imshow(fg_frame)
    ax[2].set_title("Frame sem fundo")
    ax[2].axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
import cv2
import os
import glob
import numpy as np
import matplotlib.pyplot as plt

# --------------------------------------------------
# Carrega imagens
# --------------------------------------------------
image_paths = sorted(glob.glob(os.path.join(IMAGE_FOLDER, '*.png')))
print(f"Encontradas {len(image_paths)} imagens")

# --------------------------------------------------
# Carrega ground truths
# --------------------------------------------------
all_ground_truths = tracker_utils.load_ground_truth_from_gt_txt(GT_TXT_PATH)

# --------------------------------------------------
# Frame 0 e GT
# --------------------------------------------------
frame0 = cv2.imread(image_paths[0], cv2.IMREAD_GRAYSCALE)
assert frame0 is not None

x, y, w, h = map(int, all_ground_truths[0])
H, W = frame0.shape

# --------------------------------------------------
# Máscara de fundo
# --------------------------------------------------
background_mask = np.ones((H, W), dtype=np.uint8)
background_mask[y:y+h, x:x+w] = 0

# --------------------------------------------------
# Modelo de fundo (média)
# --------------------------------------------------
background_model = frame0.astype(np.float32)
background_model[background_mask == 0] = 0
background_model /= np.maximum(background_mask, 1)
background_model = background_model.astype(np.uint8)

# --------------------------------------------------
# Loop em TODOS os frames
# --------------------------------------------------
for idx, img_path in enumerate(image_paths):

    frame = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if frame is None:
        continue

    # --------------------------------------------------
    # Remoção do fundo
    # --------------------------------------------------
    diff = cv2.absdiff(frame, background_model)

    # --------------------------------------------------
    # Otsu no residual
    # --------------------------------------------------
    _, binary = cv2.threshold(
        diff, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    # --------------------------------------------------
    # Visualização
    # --------------------------------------------------
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 3, 1)
    plt.title(f"Frame {idx}")
    plt.imshow(frame, cmap="gray")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.title("Residual (frame - fundo)")
    plt.imshow(diff, cmap="gray")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.title("Binarização (Otsu)")
    plt.imshow(binary, cmap="gray")
    plt.axis("off")

    plt.tight_layout()
    plt.show()
    plt.close()   # 🔴 ESSENCIAL para não estourar memória


### Binarização inicial

In [ ]:
def preprocess_frame_initial(frame, n_levels=100, adaptive=True):
    """
    Pré-processa um frame para entrada em modelos baseados em bits.
    Agora exibe um print do antes/depois.
    """

    # --- 1) Converter para escala de cinza somente se for RGB ---
    if len(frame.shape) == 3:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    else:
        gray = frame.copy()

    # --- 2) Normalizar [0, 1] ---
    gray_norm = gray / 255.0
    processed = gray_norm.copy()

    # --- 3) Opcional: Limiarização adaptativa ---
    if adaptive:
        mean_intensity = np.mean(gray_norm)
        _, processed = cv2.threshold(
            gray_norm, mean_intensity, 1, cv2.THRESH_BINARY
        )

    # --- PRINT DO ANTES E DEPOIS ---
    plt.figure(figsize=(10, 4))

    plt.subplot(1, 2, 1)
    plt.title("Antes (Gray)")
    plt.imshow(gray, cmap="gray")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.title("Depois (Processed)")
    plt.imshow(processed, cmap="gray")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

    # --- 4) Codificação termômetro ---
    thresholds = np.linspace(0, 1, n_levels, endpoint=False)
    thermometer = np.array([processed > t for t in thresholds], dtype=np.uint8)

    # --- 5) Achatar ---
    return thermometer.flatten()


In [ ]:
patterns = []

for i in range(0, 300, 5):   # 0, 5, 10
    frame = cv2.imread(image_paths[i], cv2.IMREAD_GRAYSCALE)

    if frame is None:
        raise ValueError(f"Erro ao carregar {image_paths[i]}")

    patch = tracker_utils.extract_patch(frame, all_ground_truths[i])

    pattern = preprocess_frame_initial(patch)

    patterns.append(pattern)

print("Processamento concluído. Vetores obtidos:", len(patterns))

### Otsu

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def preprocess_frame_otsu(frame):
    # Converte para grayscale caso seja RGB
    if len(frame.shape) == 3:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    else:
        gray = frame.copy()

    techniques = {}

    # 2) MÉDIA GLOBAL  (correção do erro)
    global_mean = int(np.mean(gray))     # precisa ser escalar Python
    no_bg_global = cv2.subtract(gray, global_mean)
    techniques["Global Mean"] = no_bg_global


    # --- Print comparativo ---
    n = len(techniques)
    plt.figure(figsize=(16, 4 * n))
    
    for i, (name, img_no_bg) in enumerate(techniques.items()):
        # --- Binarização Otsu ---
        _, otsu = cv2.threshold(
            gray, 0, 255,
            cv2.THRESH_BINARY + cv2.THRESH_OTSU
        )
    
        # === COLUNA 1 — IMAGEM ORIGINAL ===
        plt.subplot(n, 3, 3*i + 1)
        plt.title("Antes (Gray)")
        plt.imshow(gray, cmap="gray")
        plt.axis("off")
    

        # === COLUNA 3 — OTSU ===
        plt.subplot(n, 3, 3*i + 3)
        plt.title(f"{name} – Após Otsu")
        plt.imshow(otsu, cmap="gray")
        plt.axis("off")
    
    plt.tight_layout()
    plt.show()

    plt.tight_layout()
    plt.show()


In [ ]:
patterns = []

for i in range(0, 300, 5):   # 0, 5, 10
    frame = cv2.imread(image_paths[i], cv2.IMREAD_GRAYSCALE)

    if frame is None:
        raise ValueError(f"Erro ao carregar {image_paths[i]}")

    patch = tracker_utils.extract_patch(frame, all_ground_truths[i])

    pattern = preprocess_frame_otsu(patch)

    patterns.append(pattern)

print("Processamento concluído. Vetores obtidos:", len(patterns))


### Tratamento de fundo + Otsu

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def preprocess_frame(frame):
    # Converte para grayscale caso seja RGB
    if len(frame.shape) == 3:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    else:
        gray = frame.copy()

    techniques = {}

    # # 1) MÉDIA LOCAL (blur)
    # local_mean = cv2.blur(gray, (25, 25))
    # no_bg_local = cv2.subtract(gray, local_mean.astype(np.uint8))
    # techniques["Local Mean"] = no_bg_local

    # 2) MÉDIA GLOBAL  (correção do erro)
    global_mean = int(np.mean(gray))     # precisa ser escalar Python
    no_bg_global = cv2.subtract(gray, global_mean)

    # # 3) BILATERAL FILTER (preserva bordas)
    # bilateral = cv2.bilateralFilter(gray, d=15, sigmaColor=50, sigmaSpace=50)
    # no_bg_bilateral = cv2.subtract(gray, bilateral)
    # techniques["Bilateral"] = no_bg_bilateral

    # # 4) MORPHOLOGICAL TOP-HAT
    # kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (25, 25))
    # tophat = cv2.morphologyEx(gray, cv2.MORPH_TOPHAT, kernel)
    # techniques["Top-Hat"] = tophat

    # # 5) CLAHE (realce adaptativo de contraste)
    # clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    # clahe_img = clahe.apply(gray)
    # no_bg_clahe = cv2.subtract(clahe_img, gray)
    # techniques["CLAHE"] = no_bg_clahe

    # --- Print comparativo ---
    n = len(techniques)
    plt.figure(figsize=(16, 4 * n))
    
    for i, (name, img_no_bg) in enumerate(techniques.items()):
        # --- Binarização Otsu ---
        _, otsu = cv2.threshold(
            img_no_bg, 0, 255,
            cv2.THRESH_BINARY + cv2.THRESH_OTSU
        )
    
        # === COLUNA 1 — IMAGEM ORIGINAL ===
        plt.subplot(n, 3, 3*i + 1)
        plt.title("Antes (Gray)")
        plt.imshow(gray, cmap="gray")
        plt.axis("off")
    
        # === COLUNA 2 — REMOÇÃO DE FUNDO ===
        plt.subplot(n, 3, 3*i + 2)
        plt.title(f"{name} – Remoção de fundo")
        plt.imshow(img_no_bg, cmap="gray")
        plt.axis("off")
    
        # === COLUNA 3 — OTSU ===
        plt.subplot(n, 3, 3*i + 3)
        plt.title(f"{name} – Após Otsu")
        plt.imshow(otsu, cmap="gray")
        plt.axis("off")
    
    plt.tight_layout()
    plt.show()

    plt.tight_layout()
    plt.show()


In [ ]:
patterns = []

for i in range(0, 300, 5):   # 0, 5, 10
    frame = cv2.imread(image_paths[i], cv2.IMREAD_GRAYSCALE)

    if frame is None:
        raise ValueError(f"Erro ao carregar {image_paths[i]}")

    patch = tracker_utils.extract_patch(frame, all_ground_truths[i])

    pattern = preprocess_frame(patch)

    patterns.append(pattern)

print("Processamento concluído. Vetores obtidos:", len(patterns))



In [ ]:
def preprocess_frame(frame):
    # Converte para grayscale caso seja RGB
    if len(frame.shape) == 3:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    else:
        gray = frame.copy()

    techniques = {}

    # Exemplo ativado:
    # 2) MÉDIA GLOBAL (correção do erro)
    global_mean = int(np.mean(gray))
    no_bg_global = cv2.subtract(gray, global_mean)
    techniques["Global Mean"] = no_bg_global

    # --- Print comparativo ---
    n = len(techniques)

    plt.figure(figsize=(20, 4 * n))

    for i, (name, img_no_bg) in enumerate(techniques.items()):
        # --- OTSU APÓS REMOVER FUNDO ---
        _, otsu_removed = cv2.threshold(
            img_no_bg, 0, 255,
            cv2.THRESH_BINARY + cv2.THRESH_OTSU
        )

        # --- OTSU DIRETO (SEM REMOVER FUNDO) ---
        _, otsu_no_bg = cv2.threshold(
            gray, 0, 255,
            cv2.THRESH_BINARY + cv2.THRESH_OTSU
        )

        # === COLUNA 1 — IMAGEM ORIGINAL ===
        plt.subplot(n, 4, 4*i + 1)
        plt.title("Antes (Gray)")
        plt.imshow(gray, cmap="gray")
        plt.axis("off")

        # === COLUNA 2 — REMOÇÃO DE FUNDO ===
        plt.subplot(n, 4, 4*i + 2)
        plt.title(f"{name} – Remoção de fundo")
        plt.imshow(img_no_bg, cmap="gray")
        plt.axis("off")

        # === COLUNA 3 — OTSU APÓS REMOVER FUNDO ===
        plt.subplot(n, 4, 4*i + 3)
        plt.title(f"{name} – Após Otsu")
        plt.imshow(otsu_removed, cmap="gray")
        plt.axis("off")

        # === COLUNA 4 — OTSU DIRETO (SEM REMOVER FUNDO) ===
        plt.subplot(n, 4, 4*i + 4)
        plt.title("Otsu sem remover fundo")
        plt.imshow(otsu_no_bg, cmap="gray")
        plt.axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
patterns = []

for i in range(0, 300, 5):   # 0, 5, 10
    frame = cv2.imread(image_paths[i], cv2.IMREAD_GRAYSCALE)

    if frame is None:
        raise ValueError(f"Erro ao carregar {image_paths[i]}")

    patch = tracker_utils.extract_patch(frame, all_ground_truths[i])

    pattern = preprocess_frame(patch)

    patterns.append(pattern)

print("Processamento concluído. Vetores obtidos:", len(patterns))